
# Feature extraction on the unlabeled test dataset (shape, color, texture)

Reads `preprocessed_test_binary/manifest.csv`, keeping only rows from the `test` split with status `ok` or `skipped`.

For each image, loads the preprocessed image (`_preprocessed.png`: color-corrected and hair-free, without blur) and its corresponding mask (`_mask.png`) to extract up to **412 features**.

The extraction functions are defined in `features/features.py` (imported as `F`).

Saves a single CSV, `features_test.csv`, in `features_test_binary_final/`.

**Input**
- Preprocessed images: `preprocessed_test_binary/test/unlabeled/`
- Manifest: `preprocessed_test_binary/manifest.csv`

**Output**
- `features_test_binary_final/features_test.csv`

**Columns of the CSV**
- `name`: image name.
- `class`: set to `unlabeled`; no class is assigned to the test images.
- `subtype`: left empty because the subtype is unknown.
- `label`: set to `0` for every test image as a placeholder; this is not a ground-truth label.
- `qc_seg_ok` and `qc_crop_fallback`: quality flags obtained from the manifest when available.
- Feature columns, prefixed with `shape_`, `color_`, `tex_`, `gcolor_` or `gtex_`.
- `qc_errors`: extraction errors separated by `;`, empty if no errors occurred.

The `qc_*` columns are quality-control fields, not model features, and must not be used for training.

**Feature families (412 expected)**

The five feature families are identical to those used for the labeled train and validation datasets:

- `shape_` (52): lesion shape and boundary descriptors.
- `color_` (82): lesion color statistics and color asymmetry.
- `tex_` (104): lesion texture, including GLCM, LBP, LNDP and Gabor features.
- `gcolor_` (70): global image color features.
- `gtex_` (104): global image texture features.

See the feature definitions in `features/features.py` and the original feature-extraction notebook for the complete list of descriptors.

**Failures and missing values**
- If a feature family fails, its values remain NaN and the error is recorded in `qc_errors`; processing continues with the remaining images.
- If the mask is empty or degenerate (fewer than 50 pixels), lesion-dependent features remain NaN, while global color and texture features may still be extracted.
- The number of successfully extracted features may be lower than 412 for images with extraction failures.

**Important:** this notebook extracts features only. It does not predict classes or train a model.


In [1]:
# %% Imports and configuration

from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from features import features as F

# ---------------- Configuration ----------------

ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")

PROC_DIR = ROOT / "preprocessed_test_binary"
OUT_DIR = ROOT / "features_test_binary_final"

N_JOBS = 4  # -1 = all CPU cores, -2 = all but one
LIMIT = None  # Set to 20 to test a small batch; None = all images

In [2]:
# %% Load and validate the test manifest

import json
import sys

manifest_path = PROC_DIR / "manifest.csv"

if not manifest_path.is_file():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

manifest = pd.read_csv(manifest_path)

required_columns = {"name", "split", "class", "status"}
missing_columns = required_columns - set(manifest.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

manifest = manifest[
    (manifest["split"] == "test")
    & manifest["status"].isin(["ok", "skipped"])
].reset_index(drop=True)

if LIMIT is not None:
    manifest = manifest.head(LIMIT).reset_index(drop=True)

if manifest.empty:
    raise RuntimeError("No eligible test images found in the manifest.")

# Assign label 0 to every unlabeled test image
CLASS_TO_LABEL = {"unlabeled": 0}

OUT_DIR.mkdir(parents=True, exist_ok=True)

(OUT_DIR / "class_to_label.json").write_text(
    json.dumps(CLASS_TO_LABEL, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Class mapping:", CLASS_TO_LABEL)
print("Class mapping saved:", OUT_DIR / "class_to_label.json")
print("Test images selected:", len(manifest))
print(manifest["status"].value_counts().to_string())

Class mapping: {'unlabeled': 0}
Class mapping saved: C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\features_test_binary_final\class_to_label.json
Test images selected: 6340
status
ok    6340


## Extraction (parallel)

In [3]:
# %% Feature extraction (parallel)

gen = Parallel(
    n_jobs=N_JOBS,
    return_as="generator"
)(
    delayed(F.extract_one)(record, CLASS_TO_LABEL)
    for record in manifest.to_dict("records")
)

df = pd.DataFrame([
    result for result in tqdm(gen, total=len(manifest))
])

meta = [
    "name",
    "class",
    "subtype",
    "label",
    "qc_seg_ok",
    "qc_crop_fallback",
]

feat_cols = [
    col
    for prefix in F.FEATURE_PREFIXES
    for col in df.columns
    if col.startswith(prefix)
]

df["class"] = "unlabeled"
df["subtype"] = pd.NA
df["label"] = 0

for col in meta:
    if col not in df.columns:
        df[col] = pd.NA

if "qc_errors" not in df.columns:
    df["qc_errors"] = ""

df = df[meta + feat_cols + ["qc_errors"]]

print(f"{df.shape[0]} images, {len(feat_cols)} features")
print("Labels assigned:", sorted(df["label"].dropna().unique()))

  0%|          | 0/6340 [00:00<?, ?it/s]

6340 images, 412 features
Labels assigned: [np.int64(0)]


## Saving test features

In [4]:
# %% Save test features

OUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUT_DIR / "features_test.csv"

df.to_csv(output_path, index=False)

print(
    f"{output_path} -> "
    f"{df.shape[0]} rows x {df.shape[1]} columns"
)

C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\features_test_binary_final\features_test.csv -> 6340 rows x 419 columns


In [5]:
# %% Feature family summary and extraction errors

print("\nFeature family summary:")

nan_pct = df[feat_cols].isna().mean()

expected_counts = {
    "shape": 52,
    "color": 82,
    "tex": 104,
    "gcolor": 70,
    "gtex": 104,
}

for family, expected in expected_counts.items():
    cols_family = [
        c for c in feat_cols
        if c.startswith(family + "_")
    ]

    if cols_family:
        mean_nan = 100 * nan_pct[cols_family].mean()

        print(
            f"{family:6s} {len(cols_family):3d} columns "
            f"(expected {expected}) | "
            f"mean NaN: {mean_nan:.2f}%"
        )
    else:
        print(
            f"{family:6s}   0 columns "
            f"<-- check qc_errors"
        )

bad = df[
    df["qc_errors"].fillna("").ne("")
]

print(f"\nImages with at least one error: {len(bad)}")

if len(bad):
    errors = (
        bad["qc_errors"]
        .str.split(";")
        .explode()
        .dropna()
    )

    errors = errors[errors.ne("")]

    print("\nMost frequent errors:")
    print(errors.value_counts().head(10).to_string())

    print("\nFirst five affected images:")
    print(
        bad[["name", "qc_errors"]]
        .head(5)
        .to_string(index=False)
    )


Feature family summary:
shape   52 columns (expected 52) | mean NaN: 0.00%
color   82 columns (expected 82) | mean NaN: 0.00%
tex    104 columns (expected 104) | mean NaN: 0.00%
gcolor  70 columns (expected 70) | mean NaN: 0.00%
gtex   104 columns (expected 104) | mean NaN: 0.00%

Images with at least one error: 0
